# Embedding Model Comparison

This experiment tests whether a biomedical-domain embedding model improves retrieval over the baseline `all-MiniLM-L6-v2`.

The experiment deliberately keeps the corpus, chunks, queries, relevance definition, and metrics unchanged. The only major variable is the embedding model.

The comparison uses `sentence-transformers/all-MiniLM-L6-v2` as the existing baseline and `neuml/pubmedbert-base-embeddings` as the biomedical candidate. The PubMedBERT embedding model is based on PubMedBERT and is intended for semantic search over medical literature. citeturn0search3turn0search5

For this controlled experiment, retrieval is evaluated directly with cosine similarity over the encoded vectors rather than rebuilding the Qdrant collection. This isolates the embedding model from vector-database configuration.

## Hypothesis

A biomedical-domain embedding model should improve retrieval of semantically similar biomedical papers, particularly the near-miss and top-5-miss cases identified in the previous failure analysis.

The hypothesis is not that the biomedical model will necessarily win. The purpose of the experiment is to measure the trade-off rather than assume it.

## Experimental Controls

- Same 979 indexed chunks.
- Same 500 source-level queries.
- Same one-query-per-PubMed-ID evaluation approach.
- Same source-level relevance definition: matching `pubid`.
- Same metrics: Recall@1, Recall@3, Recall@5, and MRR.
- Only the embedding model changes.


## Run

Run the accompanying script from the repository root:

```bash
python src/compare_embedding_models.py
```

The script computes normalized embeddings for all chunks and queries, uses matrix multiplication as cosine similarity, ranks all chunks for each query, and reports the same retrieval metrics used by the baseline benchmark.

## Interpretation

Do not select a model based on Recall@1 alone. The relevant comparison is the overall pattern across Recall@1, Recall@3, Recall@5, and MRR.

If the biomedical model materially improves Recall@1/MRR while maintaining or improving Recall@5, that supports replacing the general-purpose baseline for this project.

If the difference is small, the baseline may remain preferable because it is smaller and faster. If the biomedical model performs worse, retain the baseline and investigate other retrieval improvements rather than changing the embedding model solely because it is domain-specific.

This experiment should be followed by targeted inspection of changed rankings, especially the 15 previously identified rank-1 failures.

## Results

The experiment produced the following results on 500 queries and 979 indexed chunks:

| Model | Dim | Recall@1 | Recall@3 | Recall@5 | MRR | Top-5 misses |
|---|---:|---:|---:|---:|---:|---:|
| `all-MiniLM-L6-v2` | 384 | 0.970 | 0.988 | 0.990 | 0.980 | 5 |
| `neuml/pubmedbert-base-embeddings` | 768 | 0.972 | 0.984 | 0.992 | 0.979 | 4 |

The biomedical model makes a small improvement at Recall@1 (+0.002) and Recall@5 (+0.002), and reduces top-5 misses from 5 to 4. However, it is slightly worse at Recall@3 (-0.004) and MRR (-0.001).

This is therefore not a decisive win for the biomedical model. It improves the chance that the correct source appears somewhere in the top five, but it does not improve ranking quality overall. The differences are small enough that we should not switch models based on aggregate metrics alone.

The biomedical model also produces 768-dimensional embeddings versus 384 dimensions for MiniLM, so it has a larger vector representation and greater computational/storage cost.

## Decision

Keep `all-MiniLM-L6-v2` as the current baseline rather than replacing it immediately.

The next experiment should inspect the 15 previously identified rank-1 failures under both models. The key question is whether the biomedical model actually fixes meaningful failure cases, or whether its small Recall@5 improvement comes from changing other rankings.

This is a more informative next step than another broad metric run because it connects the model comparison directly to the failure analysis already performed.
